# Prophet Benchmark: Python vs Go

Compares the original Python Prophet with the Go wrapper service on the **Peyton Manning Wikipedia pageviews** dataset (2,905 daily observations).

**What we measure:**
- **Fit**: Train a Prophet model (both call the same Stan binary)
- **Predict**: Generate 365-day forecast from fitted model
- **Cross-validation**: 24 simulated historical forecasts with 90-day horizon

**What we expect:**
- Fit speed: modest Go advantage (less overhead around same Stan binary)
- Predict speed: large Go advantage (pure math, no Python/numpy overhead)
- CV speed: large Go advantage (goroutine parallelism vs sequential Python)
- MAPE: near-identical (same model, same data, same Stan optimizer)

In [ ]:
import time
import json
import os
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 12})

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_PATH = os.path.join(PROJECT_DIR, "examples", "example_wp_log_peyton_manning.csv")
STAN_BINARY = os.path.join(PROJECT_DIR, "python", "prophet", "stan_model", "prophet_model.bin")

print(f"Project:  {PROJECT_DIR}")
print(f"Dataset:  {DATASET_PATH}")
print(f"Stan bin: {STAN_BINARY}")

## 1. Load and Explore the Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=["ds"])
print(f"{len(df)} rows | {df['ds'].min().date()} to {df['ds'].max().date()}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["ds"], df["y"], linewidth=0.5, alpha=0.8, color="#2563eb")
ax.set_title("Peyton Manning Wikipedia Pageviews (log scale)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("log(pageviews)")
plt.tight_layout()
plt.show()

## 2. Python Prophet Benchmark

In [ ]:
# --- Python Fit ---
py_fit_times = []
py_model = None
for i in range(3):
    m = Prophet()
    start = time.perf_counter()
    m.fit(df)
    elapsed = time.perf_counter() - start
    py_fit_times.append(elapsed)
    print(f"  fit {i+1}/3: {elapsed:.3f}s")
    if py_model is None:
        py_model = m

print(f"\n  mean fit: {np.mean(py_fit_times):.3f}s")

In [ ]:
# --- Python Predict ---
py_future = py_model.make_future_dataframe(periods=365)
py_pred_times = []
py_forecast = None
for i in range(5):
    start = time.perf_counter()
    fc = py_model.predict(py_future)
    elapsed = time.perf_counter() - start
    py_pred_times.append(elapsed)
    print(f"  predict {i+1}/5: {elapsed:.3f}s")
    if py_forecast is None:
        py_forecast = fc

print(f"\n  mean predict: {np.mean(py_pred_times):.3f}s ({len(py_future)} rows)")

In [ ]:
# --- Python Cross-Validation ---
py_cv_model = Prophet()
py_cv_model.fit(df)

start = time.perf_counter()
py_cv_results = cross_validation(py_cv_model, initial="730 days", period="90 days", horizon="90 days")
py_cv_time = time.perf_counter() - start

py_metrics = performance_metrics(py_cv_results)
py_mape = py_metrics["mape"].mean()
py_n_cutoffs = py_cv_results["cutoff"].nunique()

print(f"  cv: {py_cv_time:.3f}s ({py_n_cutoffs} cutoffs)")
print(f"  MAPE: {py_mape:.4f}")

## 3. Go Prophet Benchmark

Exports the dataset as JSON, then runs the Go benchmark binary which loads the same data, fits, predicts, and cross-validates.

In [ ]:
# Export dataset for Go to consume
records = [{"ds": row["ds"].timestamp(), "y": float(row["y"])} for _, row in df.iterrows()]
with open("/tmp/prophet_bench_data.json", "w") as f:
    json.dump(records, f)
print(f"Exported {len(records)} rows to /tmp/prophet_bench_data.json")

# Run Go benchmark
go_dir = os.path.join(PROJECT_DIR, "go")
env = os.environ.copy()
env["PROPHET_STAN_BINARY"] = STAN_BINARY

result = subprocess.run(
    ["go", "run", "./cmd/bench"],
    cwd=go_dir, env=env, capture_output=True, text=True, timeout=300,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Load Go results
with open("/tmp/prophet_bench_go.json") as f:
    go_results = json.load(f)

go_results

## 4. Speed Comparison

In [ ]:
# Assemble results
py_fit_mean = np.mean(py_fit_times)
py_pred_mean = np.mean(py_pred_times)

comparison = pd.DataFrame({
    "Metric": ["Fit (mean)", "Predict (mean)", "Cross-Validation"],
    "Python (s)": [py_fit_mean, py_pred_mean, py_cv_time],
    "Go (s)": [go_results["fit_mean_s"], go_results["predict_mean_s"], go_results["cv_time_s"]],
})
comparison["Speedup"] = comparison["Python (s)"] / comparison["Go (s)"]
comparison = comparison.set_index("Metric")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"Python": "#3b82f6", "Go": "#10b981"}

for i, metric in enumerate(comparison.index):
    ax = axes[i]
    py_val = comparison.loc[metric, "Python (s)"]
    go_val = comparison.loc[metric, "Go (s)"]
    speedup = comparison.loc[metric, "Speedup"]

    bars = ax.bar(["Python", "Go"], [py_val, go_val],
                  color=[colors["Python"], colors["Go"]], width=0.5, edgecolor="white", linewidth=1.5)

    # Add time labels on bars
    for bar, val in zip(bars, [py_val, go_val]):
        if val >= 0.01:
            label = f"{val:.3f}s"
        else:
            label = f"{val*1000:.1f}ms"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + py_val * 0.03,
                label, ha="center", va="bottom", fontsize=11, fontweight="bold")

    ax.set_title(f"{metric}\n{speedup:.0f}x faster", fontsize=13, fontweight="bold")
    ax.set_ylabel("Time (seconds)")
    ax.set_ylim(0, py_val * 1.25)

fig.suptitle("Python Prophet vs Go Prophet — Execution Time", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

comparison.style.format({"Python (s)": "{:.4f}", "Go (s)": "{:.4f}", "Speedup": "{:.1f}x"})

## 5. Python Prophet Forecast Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Historical data
ax.scatter(df["ds"], df["y"], s=1, alpha=0.4, color="#64748b", label="Observed", zorder=2)

# Forecast
fc = py_forecast
ax.plot(fc["ds"], fc["yhat"], color="#2563eb", linewidth=1.5, label="Python Prophet yhat", zorder=3)
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.15, color="#2563eb", label="80% interval")

# Mark train/forecast boundary
last_train = df["ds"].max()
ax.axvline(last_train, color="#ef4444", linestyle="--", linewidth=1, alpha=0.7, label="Forecast start")

ax.set_title("Python Prophet — Fit + 365-day Forecast", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("log(pageviews)")
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Prophet Components (Trend + Seasonality)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Trend
axes[0].plot(fc["ds"], fc["trend"], color="#2563eb", linewidth=1.5)
axes[0].fill_between(fc["ds"], fc["trend_lower"], fc["trend_upper"], alpha=0.15, color="#2563eb")
axes[0].set_title("Trend", fontsize=13, fontweight="bold")
axes[0].set_ylabel("log(pageviews)")

# Weekly seasonality
axes[1].plot(fc["ds"], fc["weekly"], color="#10b981", linewidth=1)
axes[1].set_title("Weekly Seasonality", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Effect")

# Yearly seasonality
axes[2].plot(fc["ds"], fc["yearly"], color="#f59e0b", linewidth=1)
axes[2].set_title("Yearly Seasonality", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Effect")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

## 7. Cross-Validation MAPE Comparison

In [ ]:
# MAPE by horizon (Python)
py_cv_results["horizon_days"] = (py_cv_results["ds"] - py_cv_results["cutoff"]).dt.days
py_mape_by_horizon = py_cv_results.groupby("horizon_days").apply(
    lambda g: np.mean(np.abs((g["yhat"] - g["y"]) / g["y"]))
).reset_index()
py_mape_by_horizon.columns = ["horizon_days", "mape"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAPE by horizon
ax = axes[0]
ax.plot(py_mape_by_horizon["horizon_days"], py_mape_by_horizon["mape"] * 100,
        marker="o", markersize=3, color="#2563eb", linewidth=1.5, label="Python Prophet")
ax.set_title("MAPE by Forecast Horizon", fontsize=13, fontweight="bold")
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("MAPE (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
ax.legend()

# Overall MAPE comparison
ax = axes[1]
go_mape = go_results["cv_mape"]
bars = ax.bar(["Python", "Go"], [py_mape * 100, go_mape * 100],
              color=[colors["Python"], colors["Go"]], width=0.4, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, [py_mape, go_mape]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val*100:.2f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_title("Overall CV MAPE", fontsize=13, fontweight="bold")
ax.set_ylabel("MAPE (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))

mape_diff = abs(py_mape - go_mape)
ax.annotate(f"Δ = {mape_diff*100:.2f}% ({mape_diff/py_mape*100:.1f}% relative)",
            xy=(0.5, 0.92), xycoords="axes fraction", ha="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#f0fdf4", edgecolor="#10b981"))

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
# Final summary table
fit_speedup = py_fit_mean / go_results["fit_mean_s"]
pred_speedup = py_pred_mean / go_results["predict_mean_s"]
cv_speedup = py_cv_time / go_results["cv_time_s"]

summary = pd.DataFrame({
    "": ["Fit (3 runs)", "Predict (5 runs, 3270 rows)", "Cross-Validation (24 cutoffs)", "CV MAPE"],
    "Python": [
        f"{py_fit_mean:.3f}s",
        f"{py_pred_mean:.3f}s",
        f"{py_cv_time:.3f}s",
        f"{py_mape:.4f}",
    ],
    "Go": [
        f"{go_results['fit_mean_s']:.3f}s",
        f"{go_results['predict_mean_s']:.4f}s",
        f"{go_results['cv_time_s']:.3f}s",
        f"{go_results['cv_mape']:.4f}",
    ],
    "Speedup": [
        f"{fit_speedup:.1f}x",
        f"{pred_speedup:.0f}x",
        f"{cv_speedup:.1f}x",
        f"Δ={abs(py_mape - go_mape)*100:.2f}%",
    ],
}).set_index("")

print("Dataset: Peyton Manning Wikipedia pageviews (2,905 days)")
print(f"Both use the same compiled Stan binary for model fitting.\n")
summary